# NYC-Commutable School District Analysis
This notebook contains the complete data engineering and Exploratory Data Analysis (EDA) pipeline used to build a mathematically exhaustive database of all school districts in the 14 NYC-commutable counties.

It merges two primary datasets:
1. **Stanford SEDA (V6.0)**: Provides exhaustive geographic coverage, student enrollment, absolute test scores, and academic growth metrics.
2. **Niche**: Provides commercial letter grades (A+, A, etc.) which we fuzzy-matched onto the Stanford data.

In [1]:
import pandas as pd
import requests
import os
import difflib
import re

# Suppress DtypeWarnings
import warnings
warnings.filterwarnings('ignore')

## Step 1: Download Stanford (SEDA) Datasets
We download the District-Level covariate pool, the test score pool, and the geographic crosswalk.

In [2]:
def download_file(url, local_filename):
    if not os.path.exists(local_filename):
        print(f"Downloading {url}...")
        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            with open(local_filename, 'wb') as f:
                for chunk in r.iter_content(chunk_size=8192): 
                    f.write(chunk)
    else:
        print(f"File {local_filename} already exists.")
    return local_filename

cov_url = "https://stacks.stanford.edu/file/xh833nn4025/seda_cov_admindist_pool_6.0.csv"
score_url = "https://stacks.stanford.edu/file/xh833nn4025/seda_admindist_pool_cs_6.0.csv"
crosswalk_url = "https://stacks.stanford.edu/file/xh833nn4025/seda_crosswalk_6.0.csv"

cov_file = "seda_cov.csv"
score_file = "seda_score.csv"
crosswalk_file = "seda_crosswalk.csv"

download_file(cov_url, cov_file)
download_file(score_url, score_file)
download_file(crosswalk_url, crosswalk_file)

File seda_cov.csv already exists.
File seda_score.csv already exists.
File seda_crosswalk.csv already exists.


'seda_crosswalk.csv'

## Step 2: Ingest and Filter SEDA Data
Filter the data precisely to the 14 NYC commutable counties using the Stanford geographic crosswalk.

In [3]:
print("Loading data into Pandas...")
df_cov = pd.read_csv(cov_file, dtype={'sedaadmin': str, 'fips': str})
df_score = pd.read_csv(score_file, dtype={'sedaadmin': str, 'fips': str})
df_crosswalk = pd.read_csv(crosswalk_file, dtype={'sedaadmin': str})

target_counties_nj = ['Bergen County', 'Hudson County', 'Essex County', 'Union County', 
                      'Morris County', 'Passaic County', 'Somerset County', 'Middlesex County', 
                      'Monmouth County']
target_counties_ny = ['Nassau County', 'Suffolk County', 'Westchester County', 'Rockland County']
target_counties_ct = ['Fairfield County']

# Merge on all common geographic columns
df_merged = pd.merge(df_cov, df_score, on=['sedaadmin', 'sedaadminname', 'fips', 'stateabb'], how='inner')

# Get unique county mappings from crosswalk and merge
df_cw_unique = df_crosswalk[['sedaadmin', 'countyname']].drop_duplicates()
df_merged = pd.merge(df_merged, df_cw_unique, on='sedaadmin', how='left')

# Apply exact state/county masks to prevent pulling "Union County, AR"
mask_nj = (df_merged['stateabb'] == 'NJ') & (df_merged['countyname'].isin(target_counties_nj))
mask_ny = (df_merged['stateabb'] == 'NY') & (df_merged['countyname'].isin(target_counties_ny))
mask_ct = (df_merged['stateabb'] == 'CT') & (df_merged['countyname'].isin(target_counties_ct))

df_seda = df_merged[mask_nj | mask_ny | mask_ct].drop_duplicates(subset=['sedaadmin']).copy()
print(f"Total commutable districts found: {len(df_seda)}")

# Clean up columns
cols_to_keep = [
    'stateabb', 'countyname', 'sedaadminname', 'sedaadmin', 'totenrl', 
    'cs_mn_avg_ol', 'cs_mn_lrn_ol', 'sesavgall', 'baplusavgall', 'perfrl'
]
df_seda = df_seda[cols_to_keep].rename(columns={
    'stateabb': 'State',
    'countyname': 'County',
    'sedaadminname': 'District_Name',
    'totenrl': 'Total_Enrollment',
    'cs_mn_avg_ol': 'SEDA_Absolute_Score',
    'cs_mn_lrn_ol': 'SEDA_Growth_Score',
    'sesavgall': 'SES_Index',
    'baplusavgall': 'BA_Plus_Percentage',
    'perfrl': 'Free_Reduced_Lunch_Percentage'
})

Loading data into Pandas...
Total commutable districts found: 514


## Step 3: Fuzzy Merge Niche Grades
Load the Niche scrape and fuzzy-match the district names onto the Stanford dataset.

In [4]:
def clean_district_name(name):
    if pd.isna(name): return ""
    name = str(name).lower()
    filler_words = [
        r'\bpublic\b', r'\bschools\b', r'\bschool\b', r'\bdistrict\b',
        r'\bregional\b', r'\btownship\b', r'\bborough\b', r'\bof\b', r'\bthe\b',
        r'\bboard\b', r'\beducation\b', r'\bunion\b', r'\bfree\b', r'\bcentral\b',
        r'\bcity\b', r'\bvocational\b', r'\btechnical\b', r'\bhudson county schools of technology\b'
    ]
    for w in filler_words:
        name = re.sub(w, '', name)
    name = re.sub(r'[^\w\s]', '', name)
    return re.sub(r'\s+', ' ', name).strip()

niche_file = "../brain/1aa0d3c1-7a3f-4417-9041-18ac4fb2f2a3/commutable_districts.csv"
if os.path.exists(niche_file):
    df_niche = pd.read_csv(niche_file)
    df_niche['State'] = df_niche['State'].map({'Connecticut': 'CT', 'New Jersey': 'NJ', 'New York': 'NY'})
    df_niche['County'] = df_niche['County'] + " County"

    df_seda['Clean_Name'] = df_seda['District_Name'].apply(clean_district_name)
    df_niche['Clean_Name'] = df_niche['School District Name'].apply(clean_district_name)

    df_seda['Niche_Grade'] = None
    matched_count = 0

    for idx, niche_row in df_niche.iterrows():
        seda_cands = df_seda[(df_seda['State'] == niche_row['State']) & (df_seda['County'] == niche_row['County'])]
        matches = difflib.get_close_matches(niche_row['Clean_Name'], seda_cands['Clean_Name'].tolist(), n=1, cutoff=0.7)
        if matches:
            seda_idx = seda_cands[seda_cands['Clean_Name'] == matches[0]].index[0]
            df_seda.at[seda_idx, 'Niche_Grade'] = niche_row['Niche Grade']
            matched_count += 1

    print(f"Matched {matched_count} out of {len(df_niche)} Niche districts.")
    df_seda = df_seda.drop(columns=['Clean_Name'])
else:
    print("Niche file not found.")

Matched 120 out of 131 Niche districts.


## Step 4: Final Output & EDA Preview
Sort the dataset so the most premium districts bubble to the top, and display the first 10 rows.

In [5]:
# Sort by Niche Grade then Stanford Growth Score
df_seda = df_seda.sort_values(by=['Niche_Grade', 'SEDA_Growth_Score'], ascending=[True, False], na_position='last')

# Save the final master file
output_path = "../brain/1aa0d3c1-7a3f-4417-9041-18ac4fb2f2a3/tri_state_master_with_niche.csv"
df_seda.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

# Display the top 10 results
df_seda.head(100)

Saved to ../brain/1aa0d3c1-7a3f-4417-9041-18ac4fb2f2a3/tri_state_master_with_niche.csv


,State,County,District_Name,sedaadmin,Total_Enrollment,SEDA_Absolute_Score,SEDA_Growth_Score,SES_Index,BA_Plus_Percentage,Free_Reduced_Lunch_Percentage,Niche_Grade
112404,NY,Westchester County,Pleasantville Union Free School District,3623310,803.818182,0.727512,0.104521,2.051890,0.637872,0.073932,A
106210,NJ,Passaic County,Wayne Township Public School District,3417280,3778.909091,0.522355,0.073048,1.604468,0.490888,0.088284,A
114402,NY,Rockland County,Clarkstown Central School District,3620340,3911.727273,0.471519,0.067791,1.836776,0.533814,0.102859,A
111764,NY,Rockland County,Nanuet Union Free School District,3620190,1051.090909,0.671795,0.052587,1.552971,0.448879,0.153657,A
104539,NJ,Middlesex County,Edison Township School District,3404500,6969.818182,0.858518,0.046352,1.448865,0.519495,0.173523,A
...,...,...,...,...,...,...,...,...,...,...,...
116438,NY,Nassau County,Freeport Union Free School District,3611550,2897.454545,-0.322384,-0.054207,0.108002,0.240671,0.655095,B+
105868,NJ,Hudson County,Jersey City Public Schools,3407830,11241.545455,-0.150176,0.010713,-0.037515,0.425513,0.772658,B-
111788,NY,Nassau County,Uniondale Union Free School District,3629280,2946.363636,-0.061959,0.007849,0.466010,0.250693,0.617827,B-
117282,NY,Suffolk County,Brentwood Union Free School District,3605280,7909.272727,-0.293486,0.001905,0.219308,0.137454,0.772517,B-


In [6]:
df_seda[df_seda['District_Name'].str.contains('Old Tappan', case=False, na=False)]


,State,County,District_Name,sedaadmin,Total_Enrollment,SEDA_Absolute_Score,SEDA_Growth_Score,SES_Index,BA_Plus_Percentage,Free_Reduced_Lunch_Percentage,Niche_Grade
108553,NJ,Bergen County,Old Tappan Public School District,3412180,541.090909,0.790074,0.064977,1.935731,0.595597,0.013271,None


## Step 5: Regional High School Mapping
Some New Jersey districts operate as K-8 elementary districts that funnel into shared "Regional High Schools". Niche ranks the Regional High Schools separately, while SEDA tracks the component K-8 towns. 

Since the official federal Secondary District crosswalk doesn't perfectly map these in NJ, we manually map the Niche Grade of the 4 missed Regional High Schools directly to their constituent towns as a new feature: `Niche_HS_Grade`. This allows us to see the K-8 Niche grade and the High School Niche grade side-by-side without duplicating rows.

In [7]:
# 1. Load the merged dataset
df_seda = pd.read_csv(output_path)

# 2. Rename the original Niche Grade column for clarity
df_seda = df_seda.rename(columns={'Niche_Grade': 'Niche_K8_Grade'})
df_seda['Niche_HS_Grade'] = None

# 3. Define the Regional High School to Component Town Mapping
# We map the town names (or partial names) that feed into each specific Regional HS
regional_mappings = {
    # Northern Valley Regional (Niche: A+)
    'A+': {
        'Northern Valley': ['Closter', 'Demarest', 'Harrington Park', 'Haworth', 'Northvale', 'Norwood', 'Old Tappan'],
        'Pascack Valley': ['Hillsdale', 'Montvale', 'River Vale', 'Woodcliff Lake'],
        'West Morris': ['Chester', 'Mendham', 'Washington Township'], # Note: Washington Twp in Morris
        'West Essex': ['Essex Fells', 'Fairfield', 'North Caldwell', 'Roseland']
    }
}

# 4. Apply the mappings
for hs_grade, hs_dict in regional_mappings.items():
    for hs_name, towns in hs_dict.items():
        for town in towns:
            # We use string matching to find the SEDA district row
            mask = df_seda['District_Name'].str.contains(town, case=False, na=False)
            
            # Special case for Washington Township to ensure it's the one in Morris County
            if town == 'Washington Township':
                mask = mask & (df_seda['County'] == 'Morris County')
            
            df_seda.loc[mask, 'Niche_HS_Grade'] = hs_grade

# 5. Rearrange columns so the grades are next to each other
cols = df_seda.columns.tolist()
k8_idx = cols.index('Niche_K8_Grade')
cols.insert(k8_idx + 1, cols.pop(cols.index('Niche_HS_Grade')))
df_seda = df_seda[cols]

# 6. Save and Preview
df_seda.to_csv(output_path, index=False)

# Preview the towns that received a mapped High School grade
df_seda[df_seda['Niche_HS_Grade'].notna()][['District_Name', 'Niche_K8_Grade', 'Niche_HS_Grade', 'SEDA_Growth_Score']].head(10)

,District_Name,Niche_K8_Grade,Niche_HS_Grade,SEDA_Growth_Score
64,Fairfield School District,A+,A+,0.011862
123,North Caldwell School District,NaN,A+,0.189236
127,Mendham Township School District,NaN,A+,0.131254
131,Woodcliff Lake School District,NaN,A+,0.112989
143,Northvale Public School District,NaN,A+,0.088557
147,Montvale Board of Education School District,NaN,A+,0.083865
154,Chester Township School District,NaN,A+,0.075415
158,Closter Public School District,NaN,A+,0.072427
159,Norwood Public School District,NaN,A+,0.072257
167,Old Tappan Public School District,NaN,A+,0.064977


## Step 6: Continuous Predictive Modeling (Random Forest Regressor)
We will train a **Random Forest Regressor** to predict a continuous Niche Score on a scale of 1-100. This allows us to achieve high recall by applying a flexible minimum cutoff threshold later.

In [8]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
import pandas as pd

df_seda = pd.read_csv(output_path)

# 1. Define Features and unify Target Variable
features = ['SEDA_Absolute_Score', 'SEDA_Growth_Score', 'SES_Index', 'BA_Plus_Percentage', 'Free_Reduced_Lunch_Percentage', 'Total_Enrollment']
df_valid = df_seda.dropna(subset=features).copy()

df_valid['Target_Niche_Grade'] = df_valid['Niche_HS_Grade'].combine_first(df_valid['Niche_K8_Grade'])

# Map Letter Grades to a 1-100 Scale
grade_map = {
    'A+': 98, 'A': 95, 'A-': 90,
    'B+': 88, 'B': 85, 'B-': 80,
    'C+': 78, 'C': 75, 'C-': 70,
    'D+': 68, 'D': 65, 'D-': 60,
    'F': 50
}

# Only keep rows where the target grade is valid for training
df_valid['Target_Score'] = df_valid['Target_Niche_Grade'].map(grade_map)

# Split into Training (known grades) and Inference (unknown grades)
df_train = df_valid.dropna(subset=['Target_Score']).copy()
df_infer = df_valid[df_valid['Target_Score'].isna()].copy()

X_train = df_train[features]
y_train = df_train['Target_Score']

print(f"Training on {len(df_train)} labeled districts. Predicting on {len(df_infer)} unknown districts.")

# 2. Train Random Forest Regressor & Validate
rf = RandomForestRegressor(n_estimators=100, random_state=42)

# Use Negative Mean Absolute Error for validation
cv_scores = cross_val_score(rf, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')
mae_scores = -cv_scores
print(f"\n5-Fold CV Mean Absolute Error (MAE): {np.mean(mae_scores):.2f} points (Scores: {mae_scores})")

rf.fit(X_train, y_train)

print("\nFeature Importances (What drives the 1-100 score?):")
for f, imp in sorted(zip(features, rf.feature_importances_), key=lambda x: x[1], reverse=True):
    print(f"  {f}: {imp:.3f}")

# 3. Inference
X_infer = df_infer[features]
df_infer['Predicted_Niche_Score'] = rf.predict(X_infer).round(1)

# Apply known scores to the training data so everything has a score
df_train['Predicted_Niche_Score'] = df_train['Target_Score']

# Combine back
df_all_scored = pd.concat([df_train, df_infer])

# Map predictions back to the main dataframe
if 'Predicted_Niche_Score' in df_seda.columns:
    df_seda = df_seda.drop(columns=['Predicted_Niche_Score'])

df_seda = df_seda.merge(df_all_scored[['sedaadmin', 'Predicted_Niche_Score']], on='sedaadmin', how='left')
df_seda.to_csv(output_path, index=False)


Training on 139 labeled districts. Predicting on 314 unknown districts.

5-Fold CV Mean Absolute Error (MAE): 3.09 points (Scores: [2.9875     0.79607143 1.66571429 6.27214286 3.71740741])

Feature Importances (What drives the 1-100 score?):
  SES_Index: 0.718
  Free_Reduced_Lunch_Percentage: 0.106
  SEDA_Absolute_Score: 0.077
  BA_Plus_Percentage: 0.055
  SEDA_Growth_Score: 0.029
  Total_Enrollment: 0.015


## Step 7: Apply High-Recall Cutoff
To optimize for high recall and ensure no great district is missed, we set a lower cutoff bound on our continuous 1-100 score. 
For instance, setting the cutoff to `80.0` ensures any district predicted to be at least a B- is kept in our shortlist. We then sort this high-recall list by Stanford's True Growth Score.

In [9]:
# Apply High-Recall Cutoff
MIN_ACCEPTABLE_SCORE = 80.0

# Filter for districts hitting the threshold
df_shortlist = df_seda[df_seda['Predicted_Niche_Score'] >= MIN_ACCEPTABLE_SCORE]

# Sort by Stanford Growth Score (Value-Add)
df_shortlist = df_shortlist.sort_values(by='SEDA_Growth_Score', ascending=False)

print(f"Total districts matching the >={MIN_ACCEPTABLE_SCORE} cutoff: {len(df_shortlist)}")

# Display top 15 hidden gems
cols_to_show = ['State', 'County', 'District_Name', 'Target_Niche_Grade', 'Predicted_Niche_Score', 'SEDA_Growth_Score', 'SES_Index']

# We need to map Target_Niche_Grade back into df_seda just for this display view
df_seda_view = pd.merge(df_shortlist, df_valid[['sedaadmin', 'Target_Niche_Grade']], on='sedaadmin', how='left')
df_seda_view[cols_to_show].head(15)


Total districts matching the >=80.0 cutoff: 387


,State,County,District_Name,Target_Niche_Grade,Predicted_Niche_Score,SEDA_Growth_Score,SES_Index
0,NJ,Essex County,North Caldwell School District,A+,98.0,0.189236,2.489697
1,NY,Suffolk County,Springs Union Free School District,NaN,93.0,0.177091,1.261420
2,CT,Fairfield County,Side By Side Charter School District,NaN,83.3,0.140921,0.844191
3,NJ,Morris County,Mendham Township School District,A+,98.0,0.131254,2.560613
4,NJ,Monmouth County,Little Silver Boro School District,NaN,97.9,0.127556,2.371359
5,NJ,Monmouth County,Spring Lake Heights School District,NaN,96.1,0.121804,1.410868
6,NY,Westchester County,North Salem Central School District,NaN,96.2,0.113744,2.019106
7,NJ,Bergen County,Woodcliff Lake School District,A+,98.0,0.112989,2.387971
8,NJ,Middlesex County,Highland Park Boro School District,NaN,92.6,0.108105,1.238997
9,NY,Nassau County,Elmont Union Free School District,NaN,84.1,0.106007,0.842456


## Step 8: Commutability & Family Closeness
To map the "vibe" of commutability, we calculate the absolute straight-line distance from the geographic center of each school district to Times Square. 
For the 4 specific family addresses, we utilize the OpenStreetMap (OSRM) routing engine to calculate the actual estimated driving time in minutes, accounting for bridges, tunnels, and road networks.

In [10]:
import pandas as pd
import numpy as np
import requests
from geopy.geocoders import Nominatim
from geopy.distance import geodesic
import time
from tqdm.notebook import tqdm

df_seda = pd.read_csv(output_path, dtype={'sedaadmin': str})

# 1. Geocode Target Addresses
geolocator = Nominatim(user_agent="nyc_commute_estimator")

targets = {
    'Times Square': "Times Square, Manhattan, NY",
    'Old Tappan Family': "Longview Court, Old Tappan, NJ",
    'Manhattan Family': "275 South St, Manhattan, NY",
    'Queens Family': "Bayside, Queens, NY",
    'Roslyn Family': "Oakwood Circle, Roslyn, NY"
}

target_coords = {}
print("Geocoding target addresses...")
for name, address in targets.items():
    location = geolocator.geocode(address)
    if location:
        target_coords[name] = (location.latitude, location.longitude)
        print(f"Found {name}: {target_coords[name]}")
    else:
        print(f"FAILED to find {name}")
    time.sleep(1) # Polite delay for Nominatim

# 2. Get School District Centroids from SEDA Crosswalk
print("\nCalculating school district centroids...")
cw = pd.read_csv('seda_crosswalk.csv', dtype={'sedaadmin': str})
cw['latcod'] = pd.to_numeric(cw['latcod'], errors='coerce')
cw['loncod'] = pd.to_numeric(cw['loncod'], errors='coerce')

# Average the coordinates of all schools in the district to find the population center
centroids = cw.groupby('sedaadmin')[['latcod', 'loncod']].mean().reset_index()

# Merge coordinates into main dataset
df_seda = df_seda.merge(centroids, on='sedaadmin', how='left')

# 3. Calculate Straight-Line Distance to Times Square
print("Calculating distance to Times Square...")
def calc_distance(row):
    if pd.isna(row['latcod']) or pd.isna(row['loncod']): return np.nan
    return geodesic((row['latcod'], row['loncod']), target_coords['Times Square']).miles

df_seda['Miles_To_Times_Square'] = df_seda.apply(calc_distance, axis=1).round(1)

# 4. Calculate OSRM Driving Times for Family Addresses
print("Calculating OSRM driving times... (This will take ~3 minutes)")

def get_drive_time_osrm(lat1, lon1, lat2, lon2):
    url = f"http://router.project-osrm.org/route/v1/driving/{lon1},{lat1};{lon2},{lat2}?overview=false"
    try:
        r = requests.get(url, timeout=5)
        data = r.json()
        if data['code'] == 'Ok':
            return data['routes'][0]['duration'] / 60.0
    except:
        pass
    return np.nan

family_targets = ['Old Tappan Family', 'Manhattan Family', 'Queens Family', 'Roslyn Family']

# Create columns
for ft in family_targets:
    df_seda[f'Drive_Mins_To_{ft.replace(" Family", "")}'] = np.nan

session = requests.Session()

# Iterate through districts (limit to those hitting our 80.0 cutoff to save API time)
# Wait, user wants the whole dataset scored. We'll score all 514.
for idx, row in df_seda.iterrows():
    if pd.isna(row['latcod']): continue
        
    origin = (row['latcod'], row['loncod'])
    
    for ft in family_targets:
        dest = target_coords[ft]
        mins = get_drive_time_osrm(origin[0], origin[1], dest[0], dest[1])
        df_seda.at[idx, f'Drive_Mins_To_{ft.replace(" Family", "")}'] = mins
        
    time.sleep(0.1) # Polite delay for OSRM public server

# Round the minutes
for ft in family_targets:
    col = f'Drive_Mins_To_{ft.replace(" Family", "")}'
    df_seda[col] = df_seda[col].round(0)

# Save the dataset
df_seda.to_csv(output_path, index=False)
print("\nCommutability mapping complete!")
df_seda[['District_Name', 'Miles_To_Times_Square', 'Drive_Mins_To_Old Tappan', 'Drive_Mins_To_Manhattan', 'Drive_Mins_To_Queens', 'Drive_Mins_To_Roslyn']].head(10)


Geocoding target addresses...
Found Times Square: (40.7570095, -73.9859724)
Found Old Tappan Family: (41.0281442, -73.9724328)
Found Manhattan Family: (40.7107934, -73.98769)
Found Queens Family: (38.1188767, -75.3115942)
Found Roslyn Family: (40.8038041, -73.6575887)

Calculating school district centroids...
Calculating distance to Times Square...
Calculating OSRM driving times... (This will take ~3 minutes)

Commutability mapping complete!


,District_Name,Miles_To_Times_Square,Drive_Mins_To_Old Tappan,Drive_Mins_To_Manhattan,Drive_Mins_To_Queens,Drive_Mins_To_Roslyn
0,Pleasantville Union Free School District,28.2,33.0,58.0,366.0,64.0
1,Wayne Township Public School District,19.0,42.0,42.0,336.0,63.0
2,Clarkstown Central School District,25.7,20.0,60.0,358.0,71.0
3,Nanuet Union Free School District,22.9,12.0,52.0,350.0,63.0
4,Edison Township School District,25.3,73.0,53.0,291.0,86.0
5,East Hampton Union Free School District,95.0,178.0,157.0,474.0,128.0
6,Somerset Hills Regional School District,30.8,71.0,64.0,318.0,89.0
7,Metuchen Public School District,25.0,72.0,52.0,290.0,85.0
8,Hillsborough Township Public School District,38.8,91.0,74.0,291.0,108.0
9,South Huntington Union Free School District,30.8,80.0,59.0,376.0,29.0


## Step 9: Walkability (EPA Smart Location Database)
We integrate the federal EPA Smart Location Database. To maintain absolute precision without dropping any rows, we use the NCES Geographic Relationship Files to crosswalk the 12-digit Census Block Group of every neighborhood in America directly into your overlying School District polygons. 
This exposes the neighborhood-level variance within each town: `Max_Walkability` proves if the town has a true dense downtown core, while `Avg_Walkability` proves if the rest of the town is suburban sprawl.

In [ ]:
import pandas as pd

print("Loading Walkability and Crosswalk Data...")
df_walk = pd.read_csv('tristate_walkability.csv', dtype={'GEOID20': str})
df_cross = pd.read_excel('GRF21/grf21_lea_blkgrp.xlsx', dtype={'LEAID': str, 'BLKGRP': str})

# Ensure BLKGRP has 12 digits
df_cross['BLKGRP'] = df_cross['BLKGRP'].str.zfill(12)
df_walk['GEOID20'] = df_walk['GEOID20'].str.zfill(12)

# Merge
df_merged = df_cross.merge(df_walk, left_on='BLKGRP', right_on='GEOID20', how='inner')

# Aggregate by School District (LEAID)
walk_agg = df_merged.groupby('LEAID')['NatWalkInd'].agg(['max', 'mean']).reset_index()
walk_agg.rename(columns={'max': 'Max_Walkability', 'mean': 'Avg_Walkability'}, inplace=True)
walk_agg['Max_Walkability'] = walk_agg['Max_Walkability'].round(1)
walk_agg['Avg_Walkability'] = walk_agg['Avg_Walkability'].round(1)

# Merge with main dataset
df_seda = df_seda.merge(walk_agg, left_on='sedaadmin', right_on='LEAID', how='left').drop(columns=['LEAID', 'latcod', 'loncod'], errors='ignore')

# Save Final
df_seda.to_csv(output_path, index=False)

print("\nTop 15 Most Walkable Downtowns in the Tri-State Area (by Max Walkability):")
cols = ['State', 'District_Name', 'Max_Walkability', 'Avg_Walkability', 'Drive_Mins_To_Manhattan', 'Predicted_Niche_Score']

df_walkable = df_seda.sort_values(by=['Max_Walkability', 'Avg_Walkability'], ascending=False)
df_walkable[cols].head(15)


## Step 10: Affordability & Market Momentum (Zillow ZHVI)
We mapped the Zillow Home Value Index (ZHVI) specifically for **3-Bedroom Homes** into our districts using the NCES School District-to-ZIP crosswalk.
Because many school districts span multiple ZIP codes (e.g. varying wealth tiers in a single town), we calculated the `Min`, `Max`, and `Average` metrics across all underlying neighborhoods.
- `Current_3BR_Price`: Absolute baseline affordability.
- `Growth_3yr`: Short-term post-COVID market momentum / "hotness".
- `Growth_10yr`: Long-term historical stability and equity appreciation.

In [ ]:
import pandas as pd

df_seda = pd.read_csv(output_path)

print("Top 15 Most Expensive School Districts for a 3-Bedroom Home:")
cols = ['State', 'District_Name', 'Current_3BR_Price_Mean', 'Current_3BR_Price_Min', 'Current_3BR_Price_Max', 'Growth_3yr_Mean', 'Growth_10yr_Mean', 'Predicted_Niche_Score']

df_expensive = df_seda.sort_values(by='Current_3BR_Price_Mean', ascending=False)
df_expensive[cols].head(15)
